# Phase 3 milestone — the SFT arm takes the REAL exam

**Runtime: L4 — MANDATORY.** The frozen protocol (EVAL_PROTOCOL.md) pins the
held-out benchmark to bf16 on L4; the Phase-1 baselines were measured there.
Do NOT use the A100 for this notebook.

**Cost:** ~30–60 min per seed on L4 (~3–5 units each). The run loop is
checkpointed per seed — results land on Drive as each seed finishes, and
re-running skips completed seeds. **If units are tight, stop after seed 3407**
(the pre-committed unbiased single pick — chosen as the tracer seed before any
results existed). All three seeds give the milestone error bars.

**What it does:**
1. Merges each seed's LoRA adapter into full bf16 weights (the harness needs a
   plain model, not an adapter)
2. Installs the bigcode harness **at the frozen commit** `8fc5bae`
3. Smoke: 20 problems × 1 attempt on merged seed 3407 — eyeball outputs before
   paying for full runs
4. Full frozen-protocol runs: 164 problems × 20 attempts, temp 0.2, top_p 0.95,
   bf16, `--prompt instruct` — identical to the baseline measurement
5. Prints the milestone table vs base 17.59% and the 30.4% target

This is a MILESTONE touch of the held-out set — allowed by the ground rules,
and the first time we learn how much of the exam gap SFT alone closed.

In [ ]:
# 1) GPU check — the frozen protocol requires bf16 on L4
!nvidia-smi -L
import torch
assert torch.cuda.is_bf16_supported(), 'Switch runtime to L4 (Runtime > Change runtime type).'
name = torch.cuda.get_device_name(0)
print(name)
assert 'L4' in name, f'Protocol pins the exam to L4; this is {name}. Switch runtimes.'

In [ ]:
# 2) Drive + which seeds still need running (checkpoint-resume)
from google.colab import drive
drive.mount('/content/drive')
import os, json
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
SEEDS = [3407, 42, 1234]
def met_path(seed):
    return f'{PHASE3}/metrics_sft_s{seed}.json'
todo = [s for s in SEEDS if not os.path.exists(met_path(s))]
print('seeds already measured:', [s for s in SEEDS if s not in todo])
print('seeds to run:', todo)

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y torchao   # 2026-07-18: torchao release incompatible with Colab's torch; unused by us

In [ ]:
# 3) Merge each needed adapter into full bf16 weights (harness needs plain models)
import gc
from unsloth import FastLanguageModel
for seed in todo:
    out_dir = f'/content/merged_s{seed}'
    if os.path.exists(out_dir):
        print(seed, 'already merged'); continue
    model, tok = FastLanguageModel.from_pretrained(
        f'{PHASE3}/sft_notrace_s{seed}_ep1', max_seq_length=1024,
        load_in_4bit=False, dtype=torch.bfloat16)
    model.save_pretrained_merged(out_dir, tok, save_method='merged_16bit')
    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    print('merged seed', seed, '->', out_dir)
print('all merges done')

In [ ]:
# 4) Install the harness AT THE FROZEN COMMIT (part of the ruler)
FROZEN_COMMIT = '8fc5bae6479c4fbbb28c3f8b644f6a15b3f3b5bd'
%cd /content
!rm -rf bigcode-evaluation-harness
!git clone -q https://github.com/bigcode-project/bigcode-evaluation-harness.git
%cd /content/bigcode-evaluation-harness
!git checkout -q {FROZEN_COMMIT}
ver = !git rev-parse HEAD
assert ver[0] == FROZEN_COMMIT, 'checkout failed'
print('harness pinned at', ver[0][:8])
!pip install -q -e .
!pip install -q -U transformers accelerate
# 2026-07-18: compiled torch add-ons (unused by us) break after the upgrade
!pip uninstall -y -q torchao torchaudio torchvision

In [ ]:
# 5) SMOKE on merged seed 3407: 20 problems x 1 attempt (~5 min). READ the outputs.
TASK = 'humanevalfixtests-python'
PROMPT = 'instruct'
BATCH = 16
%cd /content/bigcode-evaluation-harness
!accelerate launch main.py \
  --model /content/merged_s3407 --tasks {TASK} --prompt {PROMPT} \
  --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 1 \
  --batch_size {BATCH} --precision bf16 --limit 20 \
  --max_length_generation 2048 --allow_code_execution \
  --save_generations --save_generations_path /content/smoke_gens.json \
  --metric_output_path /content/smoke_metrics.json
import glob
path = sorted(glob.glob('/content/smoke_gens*.json'))[0]
gens = json.load(open(path))
for i in range(3):
    print('=' * 70)
    print(f'PROBLEM {i} — SFT model output:')
    print(gens[i][0][:2000])
print('=' * 70)
print(open('/content/smoke_metrics.json').read())
print('Healthy = complete fixed functions, same shape as Phase 1. Sick = prose,')
print('empty, or truncated output. If sick: STOP and report — do not run stage 6.')

In [ ]:
# 6) FULL FROZEN-PROTOCOL RUNS — one per seed, results to Drive as each finishes
%cd /content/bigcode-evaluation-harness
for seed in todo:
    print('=' * 70)
    print(f'SEED {seed}: 164 problems x 20 attempts (~30-60 min)')
    gen_path = f'{PHASE3}/gens_sft_s{seed}.json'
    m_path = met_path(seed)
    !accelerate launch main.py \
      --model /content/merged_s{seed} --tasks {TASK} --prompt {PROMPT} \
      --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 20 \
      --batch_size {BATCH} --precision bf16 \
      --max_length_generation 2048 --allow_code_execution \
      --save_generations --save_generations_path {gen_path} \
      --metric_output_path {m_path}
    print(open(m_path).read())

In [ ]:
# 7) THE MILESTONE TABLE
BASE_P1, TARGET = 17.59, 30.4
print(f"{'model':<16} pass@1    pass@10")
print(f"{'base (Phase 1)':<16} {BASE_P1:6.2f}%   23.50%")
vals = []
for seed in SEEDS:
    if not os.path.exists(met_path(seed)):
        print(f'sft seed {seed}: not run'); continue
    m = json.load(open(met_path(seed)))
    k = [x for x in m if x != 'config'][0]
    p1 = m[k].get('pass@1', 0) * 100
    p10 = m[k].get('pass@10', 0) * 100
    vals.append(p1)
    print(f"{'sft seed ' + str(seed):<16} {p1:6.2f}%   {p10:6.2f}%")
if vals:
    mean = sum(vals) / len(vals)
    print(f"\nSFT arm mean pass@1: {mean:.2f}%  ({len(vals)} seed(s))")
    print(f'vs base {BASE_P1}%: {mean - BASE_P1:+.2f} pts')
    print(f'vs OctoCoder-16B target {TARGET}%: {mean - TARGET:+.2f} pts')
    print('\nRemember: DPO and GRPO still to come — SFT is the launchpad, not the answer.')
print('\nBring the whole table back to the session.')

## Bring back to the session
1. The **smoke outputs** (healthy/sick call)
2. The **milestone table** — per-seed pass@1/pass@10 and the mean
3. How many units the runs actually burned

Then: Phase 4 — DPO, with pairs mined from the routing pass's stored samples,
initialized from `sft_notrace_s42_ep1` (highest dev pass@16, pre-committed rule).